<a href="https://colab.research.google.com/github/Harukokoko/Haruka-Nakagawa/blob/main/%E2%98%85uncomdata_results_HS10_%E6%95%B0%E5%80%A4%E8%A8%88%E7%AE%97%E3%81%BE%E3%81%A8%E3%82%81.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [22]:
import pandas as pd
import networkx as nx
import matplotlib.pyplot as plt
from typing import List, Dict, Tuple, Callable
import scipy as sp
import numpy as np
import math
import matplotlib.pyplot as plt
import seaborn as sns
import itertools
import io
import os
from google.colab import files
from google.colab import drive

In [23]:
# Google Driveをマウント
drive.mount('/content/drive')
file_path = '/content/drive/My Drive/16-research/99_Sotsuron/ex_HS10_1995_2023.xlsx'

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [24]:
# ExcelファイルをPandasデータフレームに読み込む
df = pd.read_excel(file_path)

# データフレームを確認
df.head()

,typeCode,freqCode,refPeriodId,refYear,refMonth,period,reporterCode,reporterISO,reporterDesc,flowCode,...,netWgt,isNetWgtEstimated,grossWgt,isGrossWgtEstimated,cifvalue,fobvalue,primaryValue,legacyEstimationFlag,isReported,isAggregate
0,C,A,20230101,2023,52,2023,8,ALB,Albania,X,...,0.0,False,0.0,False,NaN,256605.466,256605.466,0,False,True
1,C,A,20230101,2023,52,2023,8,ALB,Albania,X,...,0.0,False,0.0,False,NaN,47259.678,47259.678,0,False,True
2,C,A,20230101,2023,52,2023,8,ALB,Albania,X,...,0.0,False,0.0,False,NaN,88378.555,88378.555,0,False,True
3,C,A,20230101,2023,52,2023,8,ALB,Albania,X,...,0.0,False,0.0,False,NaN,4704.566,4704.566,0,False,True
4,C,A,20230101,2023,52,2023,8,ALB,Albania,X,...,0.0,False,0.0,False,NaN,831.215,831.215,0,False,True


In [25]:
#　データ数
print("列数：",df.shape[0])
print("行数：",df.shape[1])
print("国数：",len(df['reporterDesc'].unique()))

列数： 156906
行数： 47
国数： 203


In [26]:
df.isnull().sum()

,0
typeCode,0
freqCode,0
refPeriodId,0
refYear,0
refMonth,0
period,0
reporterCode,0
reporterISO,0
reporterDesc,0
flowCode,0


In [27]:
# 欠損値の割合を計算
threshold = 0.5  # 50%以上が欠損している列を削除
df = df.loc[:, df.isna().mean() < threshold]

In [28]:
df = df[df['reporterDesc'] != 'world']
df = df[df['partnerDesc'] != 'world']
# 結果を確認
print(f"Number of rows after filtering: {len(df)}")

Number of rows after filtering: 156906


In [29]:
# 必要な列のみ抽出
df = df[["refYear", "reporterDesc", "partnerDesc", "fobvalue"]]

# FOB値が0以下の場合は除外
df = df[df["fobvalue"] > 0]

# ノード名を小文字に統一
df["reporterDesc"] = df["reporterDesc"].str.lower()
df["partnerDesc"] = df["partnerDesc"].str.lower()

# 年ごとにグラフを作成
graphs_by_year = {}

for year, group in df.groupby("refYear"):
    # 有向グラフを作成
    G = nx.DiGraph()

    # エッジを一括追加
    edges = group[["reporterDesc", "partnerDesc", "fobvalue"]].to_dict(orient="records")
    G.add_edges_from([(edge["reporterDesc"], edge["partnerDesc"], {"weight": edge["fobvalue"]}) for edge in edges])

    # 年ごとのグラフを保存
    graphs_by_year[year] = G

    # 年ごとのグラフ情報を表示
    print(f"Year: {year}")
    print(f"  Number of nodes: {G.number_of_nodes()}")
    print(f"  Number of edges: {G.number_of_edges()}")

# 年ごとのグラフが保存された辞書
print(f"Total years processed: {len(graphs_by_year)}")

Year: 1995
  Number of nodes: 220
  Number of edges: 2490
Year: 1996
  Number of nodes: 218
  Number of edges: 2641
Year: 1997
  Number of nodes: 214
  Number of edges: 2735
Year: 1998
  Number of nodes: 214
  Number of edges: 2843
Year: 1999
  Number of nodes: 216
  Number of edges: 3029
Year: 2000
  Number of nodes: 226
  Number of edges: 3344
Year: 2001
  Number of nodes: 224
  Number of edges: 3495
Year: 2002
  Number of nodes: 227
  Number of edges: 3682
Year: 2003
  Number of nodes: 229
  Number of edges: 3899
Year: 2004
  Number of nodes: 229
  Number of edges: 3895
Year: 2005
  Number of nodes: 231
  Number of edges: 3974
Year: 2006
  Number of nodes: 234
  Number of edges: 4287
Year: 2007
  Number of nodes: 226
  Number of edges: 4152
Year: 2008
  Number of nodes: 232
  Number of edges: 4434
Year: 2009
  Number of nodes: 231
  Number of edges: 4431
Year: 2010
  Number of nodes: 233
  Number of edges: 4735
Year: 2011
  Number of nodes: 229
  Number of edges: 4670
Year: 2012
  N

In [30]:
def calculate_flow_with_stop_on_target(G, source, target, weight='weight'):
    """
    始点から終点までの流入量を計算し、終点に到達したら計算を終了する関数。
    また、終点から始点へのエッジを削除する。

    Parameters:
        G (nx.DiGraph): 重み付き有向グラフ
        source (str): 始点ノード
        target (str): 終点ノード
        weight (str): 重み属性の名前

    Returns:
        float: 終点への最終流入量
    """
    # 終点から始点へのエッジを削除
    if G.has_edge(target, source):
        G = G.copy()  # オリジナルを変更しないためにコピー
        G.remove_edge(target, source)

    # 初期化: 各ノードの流入量を0に設定
    flow = {node: 0 for node in G.nodes}
    flow[source] = 1  # 始点の流入量を1に設定

    visited = set()  # 訪問済みノードを記録

    def dfs(node):
        if node == target:
            return  # 終点に到達したら終了

        visited.add(node)
        out_edges = G.out_edges(node, data=True)
        total_weight = sum(edge_data[weight] for _, _, edge_data in out_edges)

        if total_weight == 0:  # 出力がない場合
            return

        for _, neighbor, edge_data in out_edges:
            proportion = edge_data[weight] / total_weight
            if neighbor not in visited:
                flow[neighbor] += flow[node] * proportion
                dfs(neighbor)  # 再帰的に隣接ノードを処理

    # DFSを開始
    dfs(source)

    return flow.get(target, 0)

In [31]:
# 基点と終点を設定
source_node = "china"
target_node = "usa"

# 年ごとの結果を保存するリスト
yearly_results = []

# 年ごとにグラフを処理
for year, G in graphs_by_year.items():

    # サブグラフ（距離4以下のノードのみを含む）
    paths_within_distance = list(nx.all_simple_paths(G, source=source_node, target=target_node, cutoff=4))
    nodes_within_paths = set(node for path in paths_within_distance for node in path)
    subgraph = G.subgraph(nodes_within_paths)

    # アメリカへの最終流入量を計算
    usa_flow = calculate_flow_with_stop_on_target(subgraph, source=source_node, target=target_node, weight="weight")

    # 直接輸出割合を計算
    if subgraph.has_edge(source_node, target_node):
        direct_weight = subgraph[source_node][target_node]["weight"]
        total_source_weight = sum(edge_data["weight"] for _, _, edge_data in subgraph.out_edges(source_node, data=True))
        direct_export_ratio = direct_weight / total_source_weight
    else:
        direct_export_ratio = 0  # 直接輸出がない場合

    # 直間比率を計算
    if usa_flow > 0:
        The_ratio_of_di = direct_export_ratio / usa_flow
        The_ratio_of_indi = 1 - The_ratio_of_di
        Indi_to_di = The_ratio_of_indi / The_ratio_of_di
    else:
        The_ratio_of_di = 0
        The_ratio_of_indi = 0
        Indi_to_di = 0

    # 中国の全輸出額を計算
    china_total_fobvalue = df[df["reporterDesc"].str.lower() == "china"]["fobvalue"].sum()

    # サブグラフ内の中国の全輸出額
    if subgraph.has_node("china"):
        china_total_export = sum(edge_data["weight"] for _, _, edge_data in subgraph.out_edges("china", data=True))
    else:
        china_total_export = 0

    # サブグラフ内の中国から米国への直接輸出額
    direct_weight = subgraph[source_node][target_node]["weight"] if subgraph.has_edge(source_node, target_node) else 0

    # アメリカへの推定流入額
    usa_final_estimation = usa_flow * china_total_export

    # 年ごとの結果を保存
    yearly_results.append({
        "Year": year,
        "Flow to USA": usa_flow,
        "Direct Export Ratio": direct_export_ratio,
        "Direct Export Amount": direct_weight,
        "Indirect to Direct Ratio": Indi_to_di,
        "China's Total FOB Value": china_total_fobvalue,
        "China's Total Export Volume": china_total_export,
        "Estimated Final Flow to USA": usa_final_estimation
    })

# データフレームに変換
results_df = pd.DataFrame(yearly_results)

# データフレームをCSVとして保存
results_df.to_csv("yearly_export_results.csv", index=False)
print("Yearly export results saved to 'yearly_export_results.csv'.")

# データフレームの表示
print(results_df.head())

Yearly export results saved to 'yearly_export_results.csv'.
   Year  Flow to USA  Direct Export Ratio  Direct Export Amount  \
0  1995     0.004641             0.004641              294397.0   
1  1996     0.000278             0.000277               32929.0   
2  1997     0.000237             0.000125              120136.0   
3  1998     0.000227             0.000227              303054.0   
4  1999     0.009772             0.009772             9225789.0   

   Indirect to Direct Ratio  China's Total FOB Value  \
0                  0.000051             5.806170e+10   
1                  0.001884             5.806170e+10   
2                  0.895962             5.806170e+10   
3                  0.000317             5.806170e+10   
4                  0.000001             5.806170e+10   

   China's Total Export Volume  Estimated Final Flow to USA  
0                 6.343266e+07                 2.944122e+05  
1                 1.187802e+08                 3.299105e+04  
2             

In [32]:
results_df.head()

,Year,Flow to USA,Direct Export Ratio,Direct Export Amount,Indirect to Direct Ratio,China's Total FOB Value,China's Total Export Volume,Estimated Final Flow to USA
0,1995,0.004641,0.004641,294397.0,0.000051,5.806170e+10,6.343266e+07,2.944122e+05
1,1996,0.000278,0.000277,32929.0,0.001884,5.806170e+10,1.187802e+08,3.299105e+04
2,1997,0.000237,0.000125,120136.0,0.895962,5.806170e+10,9.604603e+08,2.277733e+05
3,1998,0.000227,0.000227,303054.0,0.000317,5.806170e+10,1.336776e+09,3.031502e+05
4,1999,0.009772,0.009772,9225789.0,0.000001,5.806170e+10,9.440900e+08,9.225799e+06


In [33]:
import pandas as pd
import networkx as nx

# 前年比変化率を計算して追加
results_df["DiIndiRatio Change (%)"] = results_df["Indirect to Direct Ratio"].pct_change() * 100

# 変化率の絶対値が大きい年を特定（例: 変化率が50%以上の年）
threshold = 50  # しきい値を設定
significant_years = results_df[results_df["DiIndiRatio Change (%)"].abs() > threshold]["Year"].tolist()

# 結果を保存するリスト
significant_results = []

# 流量変化が大きな経路を抽出
for year in significant_years:
    # 現在の年と前の年のグラフを取得
    current_graph = graphs_by_year.get(year)
    previous_graph = graphs_by_year.get(year - 1)

    if not previous_graph:
        continue  # 前年のグラフがない場合スキップ

    # 流量（エッジ重みの合計）を計算する関数
    def calculate_edge_weights(G, source, target, cutoff=4):
        """
        指定された始点から終点までのエッジの流量を計算する関数。
        """
        edge_weights = {}
        paths = list(nx.all_simple_paths(G, source=source, target=target, cutoff=cutoff))
        for path in paths:
            total_weight = sum(G[path[i]][path[i + 1]].get("weight", 0) for i in range(len(path) - 1))
            edge_weights[tuple(path)] = total_weight
        return edge_weights

    # 現在の年と前の年の流量を計算
    current_edge_weights = calculate_edge_weights(current_graph, "china", "usa", cutoff=4)
    previous_edge_weights = calculate_edge_weights(previous_graph, "china", "usa", cutoff=4)

    # 流量の変化率を計算
    edge_weight_changes = {}
    all_paths = set(current_edge_weights.keys()).union(previous_edge_weights.keys())
    for path in all_paths:
        current_weight = current_edge_weights.get(path, 0)
        previous_weight = previous_edge_weights.get(path, 0)
        if previous_weight > 0:  # 前年の流量が0以上の場合に変化率を計算
            change_rate = ((current_weight - previous_weight) / previous_weight) * 100
        else:
            change_rate = 100 if current_weight > 0 else 0  # 無限大を避ける
        edge_weight_changes[path] = change_rate

    # 流量変化率の上位10経路を抽出
    top_10_changes = sorted(
        edge_weight_changes.items(), key=lambda x: abs(x[1]), reverse=True
    )[:10]

    # 保存
    significant_results.append({
        "Year": year,
        "DiIndiRatio Change (%)": results_df.loc[results_df["Year"] == year, "DiIndiRatio Change (%)"].values[0],
        "Top 10 Path Changes": {path: change for path, change in top_10_changes}
    })

# 最終結果を年ごとに表示
print("\nSignificant DiIndiRatio Change (%) (Top 10 Path Flow Changes):")
for result in significant_results:
    print(f"\nYear: {result['Year']} (DiIndiRatio Change (%): {result['DiIndiRatio Change (%)']:.2f}%)")
    print("Top 10 Path Flow Changes:")
    if result["Top 10 Path Changes"]:
        for path, change in result["Top 10 Path Changes"].items():
            print(f"  Path: {' -> '.join(path)}, Change Rate: {change:.2f}%")
    else:
        print("  No significant paths.")


Significant DiIndiRatio Change (%) (Top 10 Path Flow Changes):

Year: 1996 (DiIndiRatio Change (%): 3559.10%)
Top 10 Path Flow Changes:
  Path: china -> kazakhstan -> netherlands -> egypt -> usa, Change Rate: 14410.86%
  Path: china -> kazakhstan -> netherlands -> new zealand -> usa, Change Rate: 9775.22%
  Path: china -> kazakhstan -> netherlands -> southern african customs union (...1999) -> usa, Change Rate: 7967.98%
  Path: china -> italy -> southern african customs union (...1999) -> colombia -> usa, Change Rate: 4129.13%
  Path: china -> australia -> southern african customs union (...1999) -> colombia -> usa, Change Rate: 3989.65%
  Path: china -> kazakhstan -> venezuela -> usa, Change Rate: 3779.53%
  Path: china -> argentina -> ecuador -> usa, Change Rate: 3595.04%
  Path: china -> kazakhstan -> netherlands -> türkiye -> usa, Change Rate: 3564.89%
  Path: china -> kazakhstan -> türkiye -> usa, Change Rate: 3501.04%
  Path: china -> argentina -> rep. of korea -> usa, Change Ra